# 02 · Two AI agents, one catalog — governance in action

Both agents run the **same code**. The only difference is what peter granted in notebook
00. The governed tool `search_gallery` asks Lakekeeper to vend credentials for the Gold
Lance table; if the calling identity lacks `can_read_data`, Lakekeeper returns **no
credentials** and the agent is blind — the wall is in the catalog, not in app code.

In [ ]:
import sys, logging
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)  # quiet HF anon-download notice
sys.path.insert(0, '/work')
import ollama, lance
import mlib, gt, clip_util
from mlib import (NS_RAW, RAW_TABLE, NS_BRONZE, NS_SILVER, NS_GOLD, GOLD_TABLE,
                  OLLAMA_URL, CHAT_MODEL, ANALYST, CONTRACTOR, PIPELINE, get_token)
from icehelp import catalog

## The governed tool

`search_gallery` is the agent's one tool. It embeds the query with the **same CLIP model** used to build Gold, asks pylakekeeper to **vend credentials** for the Gold Lance table (`generic_tables.load(vended=True)`), and runs a nearest-vector search. **The authorization check happens right there, in the vend:** if the calling identity lacks `can_read_data` on Gold, Lakekeeper returns no credentials, the load raises, and the tool reports `AccessDenied`. The LLM never sees the data — the wall is in the catalog, not in this code.

In [ ]:
class AccessDenied(Exception): pass

def search_gallery(query, creds, k=4):
    qvec = clip_util.embed_text([query])[0]
    try:
        with gt.client(creds) as c:  # Lakekeeper vends creds only if this identity may read Gold
            t = c.generic_tables.load(NS_GOLD, GOLD_TABLE, vended=True)
    except Exception as e:
        # Lakekeeper hides a table you can't read as a 404 (NoSuchGenericTable) — that is
        # the governance denial. Anything else (connectivity, STS, storage) is an infra
        # error, not a denial: let it surface so it can't be mistaken for "access denied".
        if 'NotFound' in type(e).__name__:
            raise AccessDenied(f'{type(e).__name__}: {e}') from e
        raise
    ds = lance.dataset(t.location, storage_options=t.lance_storage_options)
    # include `_distance` explicitly (the relevance score) — also avoids a Lance
    # deprecation warning about auto-adding it.
    return ds.to_table(nearest={'column': 'vector', 'q': qvec.tolist(), 'k': k},
                       columns=['object_id', 'title', 'theme', 'caption', 'image_uri', '_distance']).to_pylist()

def compose_answer(query, hits):
    context = '\n'.join(f"- {h['title']}: {h['caption']}" for h in hits)
    prompt = (f'A user asked: {query!r}\n\nThese museum artworks were retrieved:\n{context}\n\n'
              'Answer in 2-3 sentences, referring to the specific artworks by title.')
    try:  # the chat model only writes prose; if it's missing/slow, fall back to the hits.
        ans = ollama.Client(host=OLLAMA_URL).generate(model=CHAT_MODEL, prompt=prompt).get('response', '').strip()
        if ans: return ans
    except Exception as e:
        print(f'  (chat model {CHAT_MODEL} unavailable: {type(e).__name__}; showing retrieved items)')
    return 'Retrieved: ' + '; '.join(f"{h['title']} ({h['theme']})" for h in hits)

def run_agent(query, creds):
    try:
        hits = search_gallery(query, creds)
    except AccessDenied as e:
        return {'allowed': False, 'error': str(e)}
    return {'allowed': True, 'hits': hits, 'answer': compose_answer(query, hits)}

## The analyst agent — granted `select` on Gold

In [ ]:
QUERY = 'cats and other animals'
res = run_agent(QUERY, ANALYST)
if res['allowed']:
    print('ALLOWED — credentials vended.\n')
    for h in res['hits']: print(f"  • {h['title']}  [{h['theme']}]")
    print('\nagent answer:', res['answer'])
else:
    print('DENIED:', res['error'])

## The contractor agent — same code, no grant on Gold

Lakekeeper returns **404 `NoSuchGenericTable`**, not 403: it won't even admit the table
exists to a principal that can't see it. Either way, no credentials are vended.

In [ ]:
res = run_agent(QUERY, CONTRACTOR)
if res['allowed']:
    print('ALLOWED (unexpected!)')
else:
    print('DENIED by Lakekeeper — no credentials vended:\n ', res['error'])

## The boundary runs *through* the medallion

The contractor isn't locked out of everything — it's a legitimate collaborator. It **can**
read the source layers (raw image objects, metadata, captions); Lakekeeper withholds only
the Gold embeddings. Proof — the contractor fetches a raw image object and reads the Silver
captions just fine:

In [ ]:
cat = catalog(get_token(*CONTRACTOR))
bronze = cat.load_table(f'{NS_BRONZE}.artworks').scan().to_arrow().to_pylist()
silver = cat.load_table(f'{NS_SILVER}.artwork_features').scan().to_arrow().to_pylist()
with gt.client(CONTRACTOR) as c:  # contractor CAN vend raw
    fs = gt.s3fs(c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True))
img = gt.read_object(fs, bronze[0]['image_uri'])
print(f'contractor fetched a {len(img)}-byte raw image object and read {len(silver)} Silver captions — allowed.\n')
for r in silver[:5]: print(f"  • {r['title'][:40]!r}: {r['caption'][:60]!r}")
print('\n...but Gold (the embeddings) stays invisible to it. The wall is at Gold, per-layer.')

## Try it yourself — the governed chat widget

Type any query and ask **both** agents at once. The analyst answers from Gold; the
contractor is denied. Thumbnails come from `raw.images` (operator view, for display).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Thumbnails: read raw image objects via the pipeline account (operator view).
with gt.client(PIPELINE) as c:
    _fs = gt.s3fs(c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True))
def _thumb(uri):
    try: return gt.read_object(_fs, uri) if uri else None
    except Exception: return None

box = widgets.Text(value='cats and other animals', description='Ask:',
                   layout=widgets.Layout(width='70%'))
btn = widgets.Button(description='Ask both agents', button_style='primary')
out = widgets.Output()

def _render(label, creds):
    print(f'===== {label} =====')
    res = run_agent(box.value, creds)
    if not res['allowed']:
        print('  DENIED by Lakekeeper:', res['error']); print(); return
    print('  ' + res['answer'] + '\n')
    thumbs = [widgets.Image(value=b, format='jpg', width=110)
              for b in (_thumb(h.get('image_uri')) for h in res['hits']) if b]
    if thumbs: display(widgets.HBox(thumbs))
    for h in res['hits']: print(f"   • {h['title']} [{h['theme']}]")
    print()

def on_click(_):
    with out:
        clear_output()
        _render('analyst-agent (granted Gold)', ANALYST)
        _render('contractor-agent (denied Gold)', CONTRACTOR)

btn.on_click(on_click)
display(widgets.VBox([widgets.HBox([box, btn]), out]))

---
**The takeaway:** the access wall lives in Lakekeeper's credential-vending layer, not in
application code. A denied agent gets no keys and *cannot* read the data — even running the
exact same code as the allowed one, and even for a Lance vector table.